# Profiling a PyTorch workload on a T4

Two tools, two different blind spots.

- **py-spy** samples the Python call stack. It sees your scheduler loop, request
  handling, tokenizer. It cannot see inside a PyTorch op.
- **torch.profiler** records operators and CUDA kernels. It cannot see the Python
  glue between them.

Order of use: py-spy first to find which part of the *host* is slow, then
torch.profiler to find which *op or kernel* is slow.

Run this on Colab with a T4 attached (Runtime > Change runtime type > T4 GPU).

## 1. Setup

Colab runs as root, so `py-spy` needs no `sudo` here (on a Mac it does).

In [8]:
!pip install -q py-spy

import os
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
KERNEL_PID = os.getpid()

print("torch     :", torch.__version__)
print("device    :", DEVICE)
print("gpu       :", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "-")
print("kernel pid:", KERNEL_PID)

torch     : 2.11.0+cu130
device    : cuda
gpu       : Tesla T4
kernel pid: 2847


## 2. The workload

A toy stand-in for an inference server: tokenize a batch of requests on the host,
then run a matmul on the device.

`device=DEVICE` matters. Without it `torch.randn` allocates on the CPU and the
GPU stays idle no matter what runtime you attached. Generating on the host and
calling `.cuda()` would instead add a 32 MB copy per iteration, which would
dominate the profile and teach the wrong lesson.

In [9]:
VOCAB = {f"tok{i}": i for i in range(5000)}

def tokenize(text):
    return [VOCAB[w] for w in text.split() if w in VOCAB]

def build_batch(requests):
    return [tokenize(r) for r in requests]

def forward(batch):
    x = torch.randn(len(batch), 512, 512, device=DEVICE)
    w = torch.randn(512, 512, device=DEVICE)
    return (x @ w).softmax(dim=-1)

def handle_requests(requests):
    batch = build_batch(requests)
    return forward(batch)

requests = ["tok4999 tok4998 tok4997 tok12 tok3000"] * 32

handle_requests(requests)
torch.cuda.synchronize() if DEVICE == "cuda" else None
print("ok")

ok


## 3. py-spy on the live kernel

A notebook kernel is just a long-lived Python process, so py-spy attaches to it
by PID like anything else.

`--duration` is required: attaching to a kernel means there is no process exit to
stop on. `--idle` includes threads that are blocked — a kernel waiting for your
next cell is idle, so without this flag a mistimed recording comes back empty.

To capture a specific cell instead, start the recording in one cell and run the
heavy cell while it samples.

In [10]:
# `dump` shows what every thread is doing right now. Dumping your own kernel
# only works if something other than the MainThread is busy: a `!` magic blocks
# the MainThread, so without the worker below you just see Colab's own
# subprocess machinery sitting idle.
import threading
import time

def run_load_for_dump():
    for _ in range(60):
        handle_requests(requests)

threading.Thread(target=run_load_for_dump, daemon=True).start()
time.sleep(2)

!py-spy dump --pid {KERNEL_PID}

Process 2847: /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-220b2bbe-69e2-4014-b745-f7b700c7fe0c.json
Python v3.13.15 (/usr/bin/python3.13)

Thread 2847 (idle): "MainThread"
    _poll_process (google/colab/_system_commands.py:337)
    _monitor_process (google/colab/_system_commands.py:234)
    _run_command (google/colab/_system_commands.py:204)
    _system_compat (google/colab/_system_commands.py:454)
    system (google/colab/_shell.py:96)
    <cell line: 0> (685177742.py:15)
    run_code (IPython/core/interactiveshell.py:3553)
    run_ast_nodes (IPython/core/interactiveshell.py:3473)
    run_cell_async (IPython/core/interactiveshell.py:3257)
    _pseudo_sync_runner (IPython/core/async_helpers.py:78)
    _run_cell (IPython/core/interactiveshell.py:3030)
    run_cell (IPython/core/interactiveshell.py:2976)
    run_cell (ipykernel/zmqshell.py:528)
    do_execute (ipykernel/ipkernel.py:383)
    execute_request (ipykernel/kernelbase.py:730)
    disp

In [11]:
# sample the kernel for 20s while the loop below runs
import threading

def run_load():
    for _ in range(40):
        handle_requests(requests)

threading.Thread(target=run_load).start()
!py-spy record -o pyspy.svg --pid {KERNEL_PID} --duration 20 --idle --rate 200

py-spy> Sampling process 200 times a second for 20 seconds. Press Control-C to exit.

py-spy> 1.01s behind in sampling, results may be inaccurate. Try reducing the sampling rate
py-spy> 1.40s behind in sampling, results may be inaccurate. Try reducing the sampling rate
py-spy> 1.55s behind in sampling, results may be inaccurate. Try reducing the sampling rate
py-spy> 2.00s behind in sampling, results may be inaccurate. Try reducing the sampling rate
py-spy> 2.11s behind in sampling, results may be inaccurate. Try reducing the sampling rate
py-spy> 2.22s behind in sampling, results may be inaccurate. Try reducing the sampling rate
py-spy> 2.14s behind in sampling, results may be inaccurate. Try reducing the sampling rate
py-spy> 2.24s behind in sampling, results may be inaccurate. Try reducing the sampling rate
py-spy> 2.41s behind in sampling, results may be inaccurate. Try reducing the sampling rate
py-spy> 2.52s behind in sampling, results may be inaccurate. Try reducing the sampling

Open `pyspy.svg` in a browser tab (download it from the Colab file pane).

Reading a flamegraph:

- **Width is time.** Nothing else. Height is just call depth.
- **The x-axis is not time order.** Frames are merged and sorted alphabetically.
- **Find the widest box whose children are narrow.** A wide box that hands all
  its width to one wide child is only passing time along — keep descending. Where
  the width stops being explained by children is where time is actually burned.
- **Colors are random per function name.** Warm is not hot. There is no legend
  because there is nothing to legend.

On a GPU run py-spy gets much less useful for the compute itself: the Python
thread mostly sits in a launch call or a sync while the device works. That is the
handoff point to torch.profiler.

## 4. torch.profiler with CUDA activities

Three things here are mandatory and commonly skipped.

- **Warm up before profiling.** The first CUDA matmul pays cuBLAS autotuning and
  kernel loading, easily 100x the steady-state cost.
- **`torch.cuda.synchronize()`** before trusting any wall-clock number. Kernel
  launches return immediately; `time.perf_counter()` around a matmul measures
  launch time, not execution.
- **Only a handful of steps.** `with_stack=True` is expensive and 200 iterations
  produce an unusable trace. `schedule=` skips the first steps for you.

In [12]:
from torch.profiler import profile, ProfilerActivity, schedule



ACTIVITIES = [ProfilerActivity.CPU]

if DEVICE == "cuda":

    ACTIVITIES.append(ProfilerActivity.CUDA)



for _ in range(3):

    handle_requests(requests)

if DEVICE == "cuda":

    torch.cuda.synchronize()



with profile(

    activities=ACTIVITIES,

    schedule=schedule(wait=1, warmup=1, active=3),

    record_shapes=True,

    with_stack=True,
    acc_events=True,

) as prof:

    for _ in range(5):

        handle_requests(requests)

        prof.step()



SORT_KEY = "self_cuda_time_total" if DEVICE == "cuda" else "self_cpu_time_total"

print(prof.key_averages().table(sort_by=SORT_KEY, row_limit=12))

prof.export_chrome_trace("trace.json")

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      11.729ms        95.23%      11.729ms       3.910ms             3  
                                               aten::mm         0.73%     108.057us         0.92%     135.794us      45.265us       9.152ms        74.31%       9.152ms       3.051ms             3  
         

### Reading the table

- **Self vs total.** Self is time in that frame alone; total includes children.
  `aten::matmul` shows small self time but large total because it dispatches into
  `aten::bmm`. Always hunt on self time.
- **Self CPU vs Self CUDA.** The gap is the whole point:
  - `Self CUDA` >> `Self CPU` -> compute-bound, the GPU is the limit.
  - `Self CPU` >> `Self CUDA` -> **host-bound**, Python cannot feed the device
    fast enough. This is the failure mode in real engines, where kernels are C++
    but the API server, scheduler, batching and sampling are Python.

### Reading the trace

Download `trace.json` and drop it on [ui.perfetto.dev](https://ui.perfetto.dev).
You get two rows, host and device. Read the **gaps in the GPU row**: solid bars
mean the device is busy, white space means it is waiting on Python. A
`cudaMemcpy` or sync appearing mid-stream is a forced device-to-host transfer.

Expect `aten::normal_` (the `randn`) to dominate here, with the matmul small.
That is the real finding: this workload spends its time generating random
numbers, not computing. 8.6 GFLOP on a T4 at ~8 TFLOP/s fp32 is about 1 ms.

## 5. Act on the finding

A profile tells you where time goes. It does not tell you whether that time is
work worth doing. Here it is not — the tensors are fake inputs, so hoist them out
of the loop and profile the matmul instead.

In [13]:
x = torch.randn(32, 512, 512, device=DEVICE)
w = torch.randn(512, 512, device=DEVICE)

def forward_hoisted(batch):
    return (x @ w).softmax(dim=-1)

def handle_requests_hoisted(requests):
    build_batch(requests)
    return forward_hoisted(requests)

for _ in range(3):
    handle_requests_hoisted(requests)
if DEVICE == "cuda":
    torch.cuda.synchronize()

with profile(activities=ACTIVITIES, record_shapes=True) as prof:
    for _ in range(5):
        handle_requests_hoisted(requests)

print(prof.key_averages().table(sort_by=SORT_KEY, row_limit=10))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         1.07%     227.351us        16.11%       3.425ms     684.925us      15.260ms        83.57%      18.312ms       3.662ms             5  
                                  volta_sgemm_128x64_nn         0.00%       0.000us         0.00%       0.000us       0.000us      15.260ms        83.57%      15.260ms       3.052ms             5  
         

## 6. torch.compile, measured



`torch.compile` is an optimizer, not a microscope. It makes code faster; it does

not show you where time goes, and it makes studying harder because your readable

ops are replaced by fused Triton kernels with generated names. So it comes after

you can measure, as an experiment you profile before and after.



Here Inductor will likely fuse the softmax into the matmul epilogue — a real but

modest win.



Two T4 gotchas: the first call pays 1-3 minutes of compilation (do not mistake it

for a slowdown), and the T4 is sm_75, so **fp16 works but bf16 does not**.

Observed on a T4: softmax went from 600us to 282us per call as one fused Triton
kernel, a 2.1x on that kernel but only 8.7% end to end. `aten::mm` was left
untouched at 3.052ms because Inductor logged *"Not enough SMs to use
max_autotune_gemm mode"* and fell back to cuBLAS. The T4 has 40 SMs. Expect
`torch.compile` to fuse pointwise and reduction ops on this card and never to
beat cuBLAS on a GEMM.

In [14]:
compiled_forward = torch.compile(forward_hoisted)

compiled_forward(requests)          # pays compilation here
for _ in range(3):
    compiled_forward(requests)
if DEVICE == "cuda":
    torch.cuda.synchronize()

with profile(activities=ACTIVITIES, record_shapes=True) as prof:
    for _ in range(5):
        compiled_forward(requests)

print(prof.key_averages().table(sort_by=SORT_KEY, row_limit=10))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
## Call CompiledFxGraph fvienb4u3ft36xyldtvkimqpcrqw...         0.00%       0.000us         0.00%       0.000us       0.000us      16.673ms       100.04%      16.673ms       3.335ms             5  
                                               aten::mm         1.26%     227.136us         5.53%       1.001ms     200.164us      15.259ms        91.55%      15.259ms       3.052ms             5  
         

## 7. fp16 and the tensor cores

The fp32 run leaves most of the T4 on the table. `aten::mm` takes 3.052 ms for
8.59 GFLOP, which is 2.81 TFLOP/s against an fp32 peak of roughly 8.1 TFLOP/s —
about 35%. The kernel name is the tell: `volta_sgemm_128x64_nn` is an **S**GEMM,
single precision, so the tensor cores are idle.

The T4 is sm_75: fp16 tensor cores peak near 65 TFLOP/s, and **bf16 is not
supported**. Switching dtype is one line and is worth far more here than
`torch.compile` was.

Watch the kernel name change — expect something containing `h884` or `tensorop`
instead of `sgemm`.

In [15]:
MATMUL_FLOPS = 32 * 512 * 512 * 512 * 2
PROFILED_ITERS = 5

x_fp16 = torch.randn(32, 512, 512, device=DEVICE, dtype=torch.float16)
w_fp16 = torch.randn(512, 512, device=DEVICE, dtype=torch.float16)

def forward_fp16(batch):
    return (x_fp16 @ w_fp16).softmax(dim=-1)

for _ in range(3):
    forward_fp16(requests)
if DEVICE == "cuda":
    torch.cuda.synchronize()

with profile(activities=ACTIVITIES, record_shapes=True) as prof:
    for _ in range(PROFILED_ITERS):
        forward_fp16(requests)

print(prof.key_averages().table(sort_by=SORT_KEY, row_limit=10))

mm_us = sum(e.self_device_time_total for e in prof.key_averages()
            if e.key == "aten::mm")
mm_ms_per_iter = mm_us / PROFILED_ITERS / 1000
print(f"aten::mm      {mm_ms_per_iter:.3f} ms/iter")
tflops = MATMUL_FLOPS / (mm_ms_per_iter / 1000) / 1e12
print(f"achieved      {tflops:.2f} TFLOP/s")
print(f"fp32 baseline 3.052 ms/iter -> 2.81 TFLOP/s")

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                         aten::_softmax         0.70%      57.336us         1.17%      96.324us      19.265us       2.986ms        53.44%       2.986ms     597.298us             5  
void (anonymous namespace)::softmax_warp_forward<c10...         0.00%       0.000us         0.00%       0.000us       0.000us       2.986ms        53.44%       2.986ms     597.298us             5  
         

## Checklist



1. Move tensors to `device="cuda"`; confirm with `!nvidia-smi` that memory is in use.

2. py-spy the kernel to find host-side bottlenecks.

3. Warm up, then profile a handful of steps with CUDA activities.

4. Compare `Self CPU` against `Self CUDA` to decide host-bound vs compute-bound.

5. Open the trace in Perfetto; look for GPU idle gaps.

6. Fix the finding, re-profile, and compare **sample counts or wall time** —

   never percentages, which are relative to each run's own total.

7. Only then try `torch.compile`, and re-profile to see what moved. On a T4 it
   fuses pointwise and reduction ops but falls back to cuBLAS for GEMMs
   ("Not enough SMs to use max_autotune_gemm mode").
8. Check the kernel name. An `sgemm` means the tensor cores are idle; fp16 is
   usually a bigger win than any compiler flag.